# 无训练诊断复核
读取已提交安全汇总，不读取试题、标准答案、教材或模型正文。每题三次重复仍只是一道题；96题为按历史正确性分层的病例对照样本，不能外推整体准确率。

In [ ]:
import json
from pathlib import Path
from collections import Counter
root=Path.cwd(); root=root.parent if root.name=='docs' else root
def read(name): return json.loads((root/'docs'/name).read_text(encoding='utf-8'))
manifest=read('logistics_strategy_diagnostic_manifest_20260908.safe.json')
cross=manifest['historical_cross']
assert sum(r['items'] for r in cross)==1672
assert sum(r['step120_correct'] for r in cross)==1369
assert sum(r['cpt_correct'] for r in cross)==1371
assert sum(r['improved']-r['regressed'] for r in cross)==2
large=[r for r in cross if r['choice_count']==269]
assert sum(r['items'] for r in large)==305
assert all(r['category']=='material_handling' for r in large)
assert sum(r['items']-r['cpt_correct'] for r in large)==93
assert sum(r['selected'] for r in manifest['allocation'])==96
assert all(r['selected']==r['requested'] for r in manifest['allocation'])
print('PASS: historical join totals and all six sampling cells')


In [ ]:
result=read('logistics_strategy_diagnostic_result_20260908.safe.json')
assert result['items']==len(result['rows'])==len({r['item_hash'] for r in result['rows']})==96
assert result['requests']==96*2*3*3==1728
assert not result['training'] and not result['private_content_included']
assert len(result['table'])==36
for cell in result['table']:
 subset=[r for r in result['rows'] if r['stratum']==cell['stratum'] and r['historical_correct']==cell['historical_correct']]
 assert len(subset)==cell['items']
 actual=[r['results'][cell['model']][cell['condition']] for r in subset]
 assert sum(r['correct'] for r in actual)==cell['correct']
 assert sum(r['parse_failures'] for r in actual)==cell['parse_failures']
 assert sum(r['truncated'] for r in actual)==cell['truncated']
for model in ('step120','cpt'):
 for group in ('logistika_269','logistika_other','sc_knowledge'):
  cells=[r for r in result['table'] if r['model']==model and r['stratum']==group]
  print(model,group,{c:sum(r['correct'] for r in cells if r['condition']==c) for c in ('original','permuted','deliberate')})
print('PASS: recomputed all 36 result cells from safe per-item flags')


In [ ]:
review=read('logistics_strategy_evidence_review_20260908.safe.json')
ev=read('logistics_strategy_evidence_result_20260908.safe.json')
integrity=read('logistics_strategy_source_integrity_20260908.safe.json')
structure=read('logistics_strategy_diagnostic_structure_20260908.safe.json')
assert review['reviewed']==12 and len(review['rows'])==12
assert review['book_sufficient']==0 and review['eligible']==review['supplemental_supported']==3
assert sum(r['gold_agreement'] is True for r in review['rows'])==3
assert ev['items']==3 and ev['requests']==36
assert ev['models']['step120']['closed']['correct']==0 and ev['models']['step120']['with_evidence']['correct']==2
assert ev['models']['cpt']['closed']['correct']==0 and ev['models']['cpt']['with_evidence']['correct']==1
assert all(m[c]['parse_failures']==0 for m in ev['models'].values() for c in ('closed','with_evidence'))
assert integrity['raw_item_hash_set_digest']==integrity['frozen_item_hash_set_digest']
assert structure['items_with_gold_option_text_duplicated']==6
assert all(v['strict_wrong_identical_normalized_text']==1 for v in structure['historical_duplicate_target_outcomes'].values())
print('PASS: bounded review, supplementary test, source identity and duplicate-target checks')
